# Exploratory Data Analysis


Exploratory Data Analysis on dataset 'SampleSuperstore' to find out the patterns in data and to analyse the business trends in order to determine the weak areas that needs to be worked on in order to make more profit

# Importing Libraries

In [1]:
%load_ext cudf.pandas
import dias.rewriter
import matplotlib.pyplot as plt
import seaborn as sns
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import time

In [2]:
s = time.time()

# Reading the data

In [3]:
data=pd.read_csv("/home/dias-benchmarks/notebooks/roopacalistus/retail-supermarket-store-analysis/input/superstore/SampleSuperstore.csv")

# -- STEFANOS -- Replicate Data

In [4]:
# factor = 800
factor = 800
data = pd.concat([data]*factor)
data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7995200 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Dtype
---  ------        -----
 0   Ship Mode     object
 1   Segment       object
 2   Country       object
 3   City          object
 4   State         object
 5   Postal Code   int64
 6   Region        object
 7   Category      object
 8   Sub-Category  object
 9   Sales         float64
 10  Quantity      int64
 11  Discount      float64
 12  Profit        float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1.2+ GB


In [5]:
data.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [6]:
data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7995200 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Dtype
---  ------        -----
 0   Ship Mode     object
 1   Segment       object
 2   Country       object
 3   City          object
 4   State         object
 5   Postal Code   int64
 6   Region        object
 7   Category      object
 8   Sub-Category  object
 9   Sales         float64
 10  Quantity      int64
 11  Discount      float64
 12  Profit        float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1.2+ GB


In [7]:
data.duplicated().sum()

7985223

#### So, there are 17 duplicate entries and let us drop them.

In [8]:
# STEFANOS: Disable this. It undos the data replication.
# data.drop_duplicates(inplace= True)

In [9]:
data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7995200 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Dtype
---  ------        -----
 0   Ship Mode     object
 1   Segment       object
 2   Country       object
 3   City          object
 4   State         object
 5   Postal Code   int64
 6   Region        object
 7   Category      object
 8   Sub-Category  object
 9   Sales         float64
 10  Quantity      int64
 11  Discount      float64
 12  Profit        float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1.2+ GB


# Dropping unwanted columns 

In [10]:
data.drop(["Postal Code"], axis=1,inplace= True)

In [11]:
data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7995200 entries, 0 to 9993
Data columns (total 12 columns):
 #   Column        Dtype
---  ------        -----
 0   Ship Mode     object
 1   Segment       object
 2   Country       object
 3   City          object
 4   State         object
 5   Region        object
 6   Category      object
 7   Sub-Category  object
 8   Sales         float64
 9   Quantity      int64
 10  Discount      float64
 11  Profit        float64
dtypes: float64(3), int64(1), object(8)
memory usage: 1.1+ GB


# Correlation between the data

In [12]:
data.corr(numeric_only=True)

,Sales,Quantity,Discount,Profit
Sales,1.000000,0.200795,-0.028190,0.479064
Quantity,0.200795,1.000000,0.008623,0.066253
Discount,-0.028190,0.008623,1.000000,-0.219487
Profit,0.479064,0.066253,-0.219487,1.000000


In [13]:
# STEFANOS: Disable plotting
# sns.heatmap(data.corr(), annot=True)

# EDA

### Analysing the different kinds of Shipping Modes, Segments and categories mentioned in the data

## Shipping Mode

In [14]:
data["Ship Mode"].value_counts()

Ship Mode
Standard Class    4774400
Second Class      1556000
First Class       1230400
Same Day           434400
Name: count, dtype: int64

In [15]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Ship Mode'])

## Different Segments

In [16]:
data["Segment"].value_counts()

Segment
Consumer       4152800
Corporate      2416000
Home Office    1426400
Name: count, dtype: int64

In [17]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Segment'])

## Categories of the items

In [18]:
data["Category"].value_counts()

Category
Office Supplies    4820800
Furniture          1696800
Technology         1477600
Name: count, dtype: int64

In [19]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Category'])

#### From the above plot we can conclude that Office Supplies Category has highest number of sales. Now let us see the sub-categories as well.


## Sub-categories of items

In [20]:
data["Sub-Category"].value_counts()


Sub-Category
Binders        1218400
Paper          1096000
Furnishings     765600
Phones          711200
Storage         676800
Art             636800
Accessories     620000
Chairs          493600
Appliances      372800
Labels          291200
Tables          255200
Envelopes       203200
Bookcases       182400
Fasteners       173600
Supplies        152000
Machines         92000
Copiers          54400
Name: count, dtype: int64

In [21]:
# STEFANOS: Disable plotting
# plt.figure(figsize=(15,15))
# plt.pie(data["Sub-Category"].value_counts(), labels= data["Sub-Category"].value_counts().index, autopct ="%2f")
# plt.show()

#### Here, Sub-Category with highest sale is Binder, follwed by Paper and Furnishings as second and third respectively.

In [22]:
st_profit=data.groupby(["State"])["Profit"].sum().nlargest(20)

In [23]:
st_profit

State
California       61105109.68
New York         59230838.88
Washington       26722121.36
Michigan         19570550.08
Virginia         14878360.32
Indiana          14706349.04
Georgia          13000034.64
Kentucky          8959757.28
Minnesota         8658549.92
Delaware          7981899.84
New Jersey        7818331.04
Wisconsin         6721440.32
Rhode Island      5828503.44
Maryland          5624943.04
Massachusetts     5428401.28
Missouri          5148968.40
Alabama           4629460.24
Oklahoma          3883164.80
Arkansas          3206949.68
Connecticut       2809193.44
Name: Profit, dtype: float64

In [24]:
# STEFANOS: Disable plotting
# plt.figure(figsize=(15,8))
# st_profit.plot.bar()

In [25]:
# STEFANOS: Disable plotting
# sns.lineplot(data=data, x="Discount", y= "Profit")

#### So we can see that when discount increases profit decreases

In [26]:
# STEFANOS: Disable plotting
# data.plot(kind="scatter",x="Sales",y="Profit", c="Discount", colormap="Set1",figsize=(10,10))

In this scatter plot we can clearly see that more sales does not mean more profit. It depends on discount as well. 
When Sales is high and there is low discount, the profit margin is higher.

In [27]:
data1= data.groupby("State")[["Sales","Profit"]].sum().sort_values(by="Sales", ascending=False)
# STEFANOS: Disable plotting
# data1[:].plot.bar(color = ["Green","Red"], figsize=(20,12))
# plt.title("Profit-Loss and Sales across States")
# plt.show()

California and New York generate more profit compared to the other states.

# Profit-Loss and Sales across Region

In [28]:
data1= data.groupby("Region")[["Sales","Profit"]].sum().sort_values(by="Sales", ascending=False)
# STEFANOS: Disable plotting
# data1[:].plot.bar(color = ["Blue","Red"], figsize=(10,7))
# plt.title("Profit-Loss and Sales across Region")
# plt.show()

In [29]:
e = time.time()
print(e-s)

22.75467538833618





# Conclusion:


1. The western region generates highest profit.
2. California, NewYork and Washington generates the most sales compared to the other places. 
3. The central region generates lowest profit. 
4. Texas, Pennsylvenia, Florida, Illinois, Ohio and some other states are generating loss with high sale. So we need to give some attention towards them.

Therefore, we have to work more on California and New York. Increase the sales in these states by reducing sales in states like Texas, Florida, Ohio. By decreasing the discount rates in Central region we can increase the profit. Finally we should increase the sale of Office Supplies category as they contribute more.



